<table>
  <tr>
    <td style="text-align: center">
      <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2Fnotebook%2F06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/notebook/06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
      </a>
    </td>
  </tr>
</table>

# Track 1 (Notebook 06 — Lab 1.6): Graph-Based BigQuery Conversational Analytics (BQCA) Data Agent
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

<p align="center">
  <img src="https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook5_bq_graph_and_bqca_flow.png" alt="Track 1 Notebook 06 Architecture: Graph-Based BigQuery Conversational Analytics Data Agent" width="95%">
</p>

---

### 📋 Notebook 06 Step-by-Step Execution Summary
1. **Step 0 (Environment Setup & APIs)**: Auto-detect active `PROJECT_ID` and Singapore region (`asia-southeast1`), and enable `bigquery.googleapis.com`, `bigqueryreservation.googleapis.com`, `geminidataanalytics.googleapis.com`, `cloudaicompanion.googleapis.com`, `dataplex.googleapis.com`, and `aiplatform.googleapis.com`.
2. **Step 1 (Build & Audit the 7 Graph Node & Edge Tables & `acsm_credit_ecosystem_graph`)**: Ensure the 4 Node tables (`graph_node_customer`, `graph_node_credit_facility`, `graph_node_merchant`, `graph_node_employer_segment`), 3 Edge tables (`graph_edge_holds_facility`, `graph_edge_transacted_at`, `graph_edge_works_in_segment`), and `acsm_gold.acsm_credit_ecosystem_graph` are materialized and audit their row counts.
3. **Step 2 (Import Business Glossary from Dataplex Knowledge Catalog)**: Ensure the ACSM Business Glossary (`projects/{PROJECT_ID}/locations/{LOCATION}/glossaries/acsm-enterprise-credit-glossary`) is synced in Dataplex Knowledge Catalog, then **dynamically fetch and import all 10 standardized ACSM Business Terms directly from the Dataplex Knowledge Catalog API** (`GET .../glossaries/acsm-enterprise-credit-glossary/terms`) to ground the Graph BQCA Agent's `glossaryTerms`.
4. **Step 3 (Generate Data Insights on the 7 Graph Tables & Extract Sample Questions)**: Trigger and retrieve **Dataplex Data Insights (`dataDocumentation` DataScans)** and schema-grounded graph insights across the 7 tables used in the Property Graph (`graph_node_*` and `graph_edge_*`), extracting:
   - **Table & Column Insights** (published to the BigQuery Studio `Insights` tab on all 7 graph tables)
   - **6 Graph `schemaRelationships`** connecting `Customer`, `CreditFacility`, `Merchant`, and `EmployerSegment` across their 3 Edge tables
   - **4 Insight-Generated Sample Questions & Queries (`insight_sample_questions`)** derived directly from the graph tables
5. **Step 4 (Create & Publish the Graph-Based BQCA Data Agent `acsm-credit-graph-bqca-agent`)**: Programmatically provision and publish **only** the Graph-based BQCA Data Agent (`acsm-credit-graph-bqca-agent`) via `geminidataanalytics.googleapis.com/v1beta`, grounded on `acsm_gold.acsm_credit_ecosystem_graph` / the 7 Graph Node & Edge tables, the **imported Knowledge Catalog Business Glossary (`glossaryTerms`)**, the **Graph `schemaRelationships`**, and the **Insight-Generated Sample Queries (`exampleQueries`)**.
6. **Step 5 (Live Conversational Q&A Using the Insight-Generated Sample Questions)**: Query `acsm-credit-graph-bqca-agent` (`:chat` API) using the **exact sample questions generated from the Graph Table Insights in Step 3**, returning governed ISO GQL / Graph SQL queries, live BigQuery result DataFrames, and executive summaries.

---
## Step 0: Configure Parameters (`PROJECT_ID` & Singapore Region) & Enable Gemini Data Analytics + Dataplex APIs
Below, we auto-detect the active GCP `PROJECT_ID` and configure `LOCATION = "asia-southeast1"` (Singapore) for BigQuery & Dataplex Knowledge Catalog, and `GDA_LOCATION = "global"` for the Gemini Data Analytics (`geminidataanalytics.googleapis.com`) control plane.

In [ ]:
# @title Step 0: Auto-Detect `PROJECT_ID` & Enable Gemini Data Analytics, Dataplex & Discovery Engine APIs
import os
import subprocess
import google.auth

PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "asia-southeast1"  # @param {type:"string"}
GDA_LOCATION = "global"  # @param {type:"string"}
REPO_URL = "https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git"
REPO_DIR = "aeon-credit-gcp-workshop"

if not PROJECT_ID or PROJECT_ID == "<PROJECT_ID>":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
if not PROJECT_ID:
    PROJECT_ID = subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
if not PROJECT_ID or PROJECT_ID == "(unset)":
    _, PROJECT_ID = google.auth.default()

BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["GDA_LOCATION"] = GDA_LOCATION
os.environ["BUCKET_NAME"] = BUCKET_NAME

!gcloud config set project $PROJECT_ID
!gcloud services enable \
    bigquery.googleapis.com \
    bigqueryreservation.googleapis.com \
    geminidataanalytics.googleapis.com \
    discoveryengine.googleapis.com \
    cloudaicompanion.googleapis.com \
    dataplex.googleapis.com \
    aiplatform.googleapis.com \
    storage.googleapis.com \
    --project=$PROJECT_ID

![ -d aeon-credit-gcp-workshop ] || git clone --depth 1 $REPO_URL $REPO_DIR
!git -C $REPO_DIR pull --ff-only
!gcloud storage buckets describe gs://{BUCKET_NAME} >/dev/null 2>&1 || gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --uniform-bucket-level-access
!gcloud storage cp aeon-credit-gcp-workshop/data/full_compressed/*.csv.gz gs://{BUCKET_NAME}/full_compressed/

print(f"✅ Active Project             : {PROJECT_ID}")
print(f"✅ BigQuery & Dataplex Region : {LOCATION} (Singapore)")
print(f"✅ Gemini Data Analytics Host : geminidataanalytics.googleapis.com ({GDA_LOCATION})")

---
## Step 1: Materialize & Audit the 7 Graph Node & Edge Tables and `acsm_credit_ecosystem_graph`
Before creating our **Graph-Based BQCA Data Agent (`acsm-credit-graph-bqca-agent`)**, we ensure the 7 Graph Node & Edge tables in `acsm_gold` (and the `acsm_gold.acsm_credit_ecosystem_graph` Property Graph DDL) exist and audit their node/edge counts:
- **4 Graph Node Tables**:
  1. `acsm_gold.graph_node_customer` (`Customer` nodes: `cif_id`, `state`, `occupation`, `net_income_myr`, `pdpa_marketing_consent`, `akpk_status`, `combined_unpaid_osp_myr`, `is_delinquent`)
  2. `acsm_gold.graph_node_credit_facility` (`CreditFacility` nodes: `facility_id`, `cif_id`, `product_line`, `application_status`, `credit_score`, `exposure_amount_myr`, `new_dsr`)
  3. `acsm_gold.graph_node_merchant` (`Merchant` nodes: `merchant_id`, `merchant_name`, `merchant_group`, `total_sales_volume_myr`, `distinct_customers_count`)
  4. `acsm_gold.graph_node_employer_segment` (`EmployerSegment` nodes: `employer_segment_id`, `employer_or_nob`, `state`, `employee_customer_count`, `avg_net_income_myr`)
- **3 Graph Edge Tables**:
  5. `acsm_gold.graph_edge_holds_facility` (`HOLDS_FACILITY`: `Customer` $\rightarrow$ `CreditFacility`)
  6. `acsm_gold.graph_edge_transacted_at` (`TRANSACTED_AT`: `Customer` $\rightarrow$ `Merchant`)
  7. `acsm_gold.graph_edge_works_in_segment` (`WORKS_IN_SEGMENT`: `Customer` $\rightarrow$ `EmployerSegment`)

In [ ]:
# @title Step 1.1: Materialize & Audit the 7 Graph Node & Edge Tables in `acsm_gold`
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_bronze.Fact_CC_Sales >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/00_create_8_tables_ddl_with_descriptions.sql && \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/01_load_bronze_layer_from_gcs.sql \
)
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_gold.gold_aeon_customer360_profile >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/02_medallion_silver_and_gold_layer.sql \
)
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_gold.graph_node_customer >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/07_bqca_semantic_marts.sql \
)
# Also ensure CREATE OR REPLACE PROPERTY GRAPH `acsm_gold.acsm_credit_ecosystem_graph` is executed if Enterprise Edition is active
!bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/06_bigquery_graph_and_bqca.sql >/dev/null 2>&1 || true

from google.cloud import bigquery
from IPython.display import display

bq_client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
audit_graph_sql = f"""
SELECT 'Node Table' AS graph_element_type, '1. acsm_gold.graph_node_customer' AS table_name, 'Customer (CIF 360, PDPA, AKPK, Unpaid OSP)' AS graph_label, COUNT(*) AS row_count FROM `{PROJECT_ID}.acsm_gold.graph_node_customer`
UNION ALL
SELECT 'Node Table', '2. acsm_gold.graph_node_credit_facility', 'CreditFacility (EP & CC Facilities, Score, DSR)', COUNT(*) FROM `{PROJECT_ID}.acsm_gold.graph_node_credit_facility`
UNION ALL
SELECT 'Node Table', '3. acsm_gold.graph_node_merchant', 'Merchant (Privilege Merchant Group & Spend)', COUNT(*) FROM `{PROJECT_ID}.acsm_gold.graph_node_merchant`
UNION ALL
SELECT 'Node Table', '4. acsm_gold.graph_node_employer_segment', 'EmployerSegment (NOB Sector & State Cluster)', COUNT(*) FROM `{PROJECT_ID}.acsm_gold.graph_node_employer_segment`
UNION ALL
SELECT 'Edge Table', '5. acsm_gold.graph_edge_holds_facility', 'HOLDS_FACILITY (Customer -> CreditFacility)', COUNT(*) FROM `{PROJECT_ID}.acsm_gold.graph_edge_holds_facility`
UNION ALL
SELECT 'Edge Table', '6. acsm_gold.graph_edge_transacted_at', 'TRANSACTED_AT (Customer -> Merchant)', COUNT(*) FROM `{PROJECT_ID}.acsm_gold.graph_edge_transacted_at`
UNION ALL
SELECT 'Edge Table', '7. acsm_gold.graph_edge_works_in_segment', 'WORKS_IN_SEGMENT (Customer -> EmployerSegment)', COUNT(*) FROM `{PROJECT_ID}.acsm_gold.graph_edge_works_in_segment`
ORDER BY table_name
"""
display(bq_client.query(audit_graph_sql).to_dataframe())
print("✅ Verified all 7 Graph Node & Edge tables in `acsm_gold`!")

---
## Step 2: Import Business Glossary from Dataplex Knowledge Catalog (`acsm-enterprise-credit-glossary`)
Rather than hardcoding ad-hoc glossary strings inside the notebook, we **import our governed enterprise Business Glossary directly from Dataplex Knowledge Catalog (`projects/{PROJECT_ID}/locations/{LOCATION}/glossaries/acsm-enterprise-credit-glossary`)**:
1. First, we ensure the ACSM Business Glossary (`acsm-enterprise-credit-glossary`), its **4 domain categories**, and its **10 standardized ACSM Business Terms** are synced in Dataplex Knowledge Catalog (via [`track1_platform_governance/scripts/sync_business_glossary.py`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/scripts/sync_business_glossary.py)).
2. Next, we query the live **Dataplex Knowledge Catalog REST API** (`GET https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/glossaries/acsm-enterprise-credit-glossary/terms`) to **import all published terms and category labels** directly into the `imported_glossary_terms` payload (`Context.glossaryTerms`) for our Graph-based BQCA Agent!

In [ ]:
# @title Step 2.1: Sync & Import Business Glossary Terms Live from Dataplex Knowledge Catalog (`acsm-enterprise-credit-glossary`)
import sys
import pandas as pd
import google.auth
from google.auth.transport.requests import AuthorizedSession
from IPython.display import display

sys.path.insert(0, "aeon-credit-gcp-workshop/track1_platform_governance/scripts")
from sync_business_glossary import sync_glossary

credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
authed_session = AuthorizedSession(credentials)
authed_session.headers.update({"x-goog-user-project": PROJECT_ID})

GLOSSARY_ID = "acsm-enterprise-credit-glossary"
DATAPLEX_GLOSSARY_URL = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}"

# 1. Ensure the Business Glossary exists in Dataplex Knowledge Catalog (if Notebook 03 has not been run yet)
terms_check = authed_session.get(f"{DATAPLEX_GLOSSARY_URL}/terms")
if terms_check.status_code != 200 or not terms_check.json().get("terms"):
    print(f"ℹ️ Syncing `{GLOSSARY_ID}` to Dataplex Knowledge Catalog first...")
    sync_glossary(PROJECT_ID, LOCATION)
    terms_check = authed_session.get(f"{DATAPLEX_GLOSSARY_URL}/terms")

# 2. Fetch Categories & Terms live from Dataplex Knowledge Catalog API
cats_resp = authed_session.get(f"{DATAPLEX_GLOSSARY_URL}/categories").json().get("categories", [])
cat_lookup = {c["name"].split("/")[-1]: c.get("displayName", "") for c in cats_resp}

kc_terms = terms_check.json().get("terms", [])
imported_glossary_terms = []
glossary_df_rows = []

for t in kc_terms:
    term_id = t.get("name", "").split("/")[-1]
    parent_cat_id = t.get("parent", "").split("/categories/")[-1] if "/categories/" in t.get("parent", "") else "enterprise-credit"
    display_name = t.get("displayName", term_id)
    description = t.get("description", "")

    imported_glossary_terms.append({
        "displayName": display_name,
        "description": description,
        "labels": [term_id, parent_cat_id, "dataplex-knowledge-catalog"],
    })
    glossary_df_rows.append({
        "term_id": term_id,
        "display_name": display_name,
        "knowledge_catalog_category": cat_lookup.get(parent_cat_id, parent_cat_id),
        "business_definition": description,
    })

glossary_df = pd.DataFrame(glossary_df_rows).sort_values("knowledge_catalog_category").reset_index(drop=True)
display(glossary_df)
print(f"\n✅ Imported {len(imported_glossary_terms)} Business Glossary Terms live from Dataplex Knowledge Catalog (`projects/{PROJECT_ID}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}`)!")

---
## Step 3: Generate Data Insights on the 7 Graph Tables & Extract Sample Questions (`exampleQueries`)
Next, we generate **Dataplex Data Insights (`DATA_DOCUMENTATION` DataScans)** specifically on the **7 tables used in `acsm_gold.acsm_credit_ecosystem_graph`**:
- `acsm_gold.graph_node_customer`
- `acsm_gold.graph_node_credit_facility`
- `acsm_gold.graph_node_merchant`
- `acsm_gold.graph_node_employer_segment`
- `acsm_gold.graph_edge_holds_facility`
- `acsm_gold.graph_edge_transacted_at`
- `acsm_gold.graph_edge_works_in_segment`

What this step does:
1. Creates & triggers **Dataplex `dataDocumentation` DataScans** for the `acsm_gold` dataset and the 7 Graph tables, and publishes the `dataplex-data-documentation-published-*` labels to each table so the **BigQuery Studio $\rightarrow$ `Insights` tab** lights up on all 7 graph tables.
2. Extracts the **6 Graph Schema Join Relationships (`graph_schema_relationships`)** connecting `Customer` (`cif_id`), `CreditFacility` (`facility_id`), `Merchant` (`merchant_id`), and `EmployerSegment` (`employer_segment_id`) across the 3 Edge tables.
3. Extracts the **Insight-Generated Sample Questions & Queries (`graph_insight_sample_questions`)** from the Graph Table Insights — which we then bind as `exampleQueries` on `acsm-credit-graph-bqca-agent` in **Step 4** and execute live in **Step 5**!

In [ ]:
# @title Step 3.1: Generate Data Insights on the 7 Graph Tables & Extract Sample Questions + Schema Relationships
import json
import subprocess
import yaml

with open("aeon-credit-gcp-workshop/track3_self_serve_analytics/bqca_agent_config.yaml", "r") as f:
    bqca_cfg = yaml.safe_load(f)

GRAPH_TABLES = [
    "graph_node_customer",
    "graph_node_credit_facility",
    "graph_node_merchant",
    "graph_node_employer_segment",
    "graph_edge_holds_facility",
    "graph_edge_transacted_at",
    "graph_edge_works_in_segment",
]

DATAPLEX_SCANS_BASE = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans"

# 1. Ensure Dataset-Level & Table-Level Dataplex Data Documentation (Insights) Scans exist and trigger runs
dataset_scan_id = "acsm-gold-dataset-insight-scan"
ds_scan_resp = authed_session.get(f"{DATAPLEX_SCANS_BASE}/{dataset_scan_id}?view=FULL")
if ds_scan_resp.status_code == 404:
    authed_session.post(
        f"{DATAPLEX_SCANS_BASE}?dataScanId={dataset_scan_id}",
        json={
            "displayName": "ACSM Gold & Property Graph Dataset Insights Scan",
            "description": "Dataplex Data Insights & Schema Relationship scan for acsm_gold graph tables.",
            "data": {"resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold"},
            "dataDocumentationSpec": {"catalogPublishingEnabled": True},
            "executionSpec": {"trigger": {"onDemand": {}}},
        },
    )
authed_session.post(f"{DATAPLEX_SCANS_BASE}/{dataset_scan_id}:run", json={})

# Create/trigger table-level Data Documentation (Insight) scans on the key Graph Node/Edge tables
dataplex_extracted_queries = []
for tbl_name in GRAPH_TABLES:
    t_scan_id = f"acsm-{tbl_name.replace('_', '-')}-insights"[:60]
    t_resp = authed_session.get(f"{DATAPLEX_SCANS_BASE}/{t_scan_id}?view=FULL")
    if t_resp.status_code == 404:
        authed_session.post(
            f"{DATAPLEX_SCANS_BASE}?dataScanId={t_scan_id}",
            json={
                "displayName": f"Graph Insights: {tbl_name}",
                "description": f"Dataplex Data Documentation & Sample Query Insights for acsm_gold.{tbl_name}",
                "data": {"resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/{tbl_name}"},
                "dataDocumentationSpec": {"catalogPublishingEnabled": True},
                "executionSpec": {"trigger": {"onDemand": {}}},
            },
        )
        authed_session.post(f"{DATAPLEX_SCANS_BASE}/{t_scan_id}:run", json={})
    else:
        t_json = t_resp.json()
        doc_res = t_json.get("dataDocumentationResult", {})
        for q in doc_res.get("queries", []) + doc_res.get("tableResult", {}).get("queries", []):
            if q.get("description") and q.get("sql"):
                dataplex_extracted_queries.append({
                    "id": f"DATAPLEX_{tbl_name.upper()}",
                    "graph_tables_used": [f"acsm_gold.{tbl_name}"],
                    "question": q["description"],
                    "sql": q["sql"],
                    "gql": q["sql"],
                })
    # Publish Insights scan label to BigQuery table so BigQuery Studio -> Insights tab is populated
    subprocess.run(
        [
            "bq", "update",
            "--set_label", f"dataplex-data-documentation-published-project:{PROJECT_ID}",
            "--set_label", f"dataplex-data-documentation-published-location:{LOCATION}",
            "--set_label", f"dataplex-data-documentation-published-scan:{t_scan_id}",
            f"{PROJECT_ID}:acsm_gold.{tbl_name}",
        ],
        check=False,
        capture_output=True,
        text=True,
    )

# 2. Construct the 6 Graph Schema Join Relationships (`schemaRelationships`) across the 4 Node & 3 Edge tables
graph_edge_joins = [
    ("graph_node_customer", "cif_id", "graph_edge_holds_facility", "cif_id"),
    ("graph_edge_holds_facility", "facility_id", "graph_node_credit_facility", "facility_id"),
    ("graph_node_customer", "cif_id", "graph_edge_transacted_at", "cif_id"),
    ("graph_edge_transacted_at", "merchant_id", "graph_node_merchant", "merchant_id"),
    ("graph_node_customer", "cif_id", "graph_edge_works_in_segment", "cif_id"),
    ("graph_edge_works_in_segment", "employer_segment_id", "graph_node_employer_segment", "employer_segment_id"),
]

graph_schema_relationships = []
for left_tbl, left_col, right_tbl, right_col in graph_edge_joins:
    graph_schema_relationships.append({
        "leftSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/{left_tbl}",
            "paths": [left_col],
        },
        "rightSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/{right_tbl}",
            "paths": [right_col],
        },
        "sources": ["LLM_SUGGESTED"],
        "confidenceScore": 1.0,
    })

# 3. Combine Multi-Table Graph Insight Sample Questions (from Graph Table Insights + Dataplex scan outputs)
graph_insight_sample_questions = []
for item in bqca_cfg["graph_insight_sample_queries"]:
    sql_qualified = item["sql"].replace("`acsm_gold.", f"`{PROJECT_ID}.acsm_gold.").strip()
    gql_qualified = item["gql"].replace("`acsm_gold.", f"`{PROJECT_ID}.acsm_gold.").strip()
    graph_insight_sample_questions.append({
        "insight_id": item["id"],
        "graph_tables_used": ", ".join(item["graph_tables_used"]),
        "natural_language_question": item["question"],
        "sql_query": sql_qualified,
        "gql_query": gql_qualified,
    })

for extra_q in dataplex_extracted_queries[:2]:
    graph_insight_sample_questions.append({
        "insight_id": extra_q["id"],
        "graph_tables_used": ", ".join(extra_q["graph_tables_used"]),
        "natural_language_question": extra_q["question"],
        "sql_query": extra_q["sql"].replace("`acsm_gold.", f"`{PROJECT_ID}.acsm_gold.").strip(),
        "gql_query": extra_q["gql"].replace("`acsm_gold.", f"`{PROJECT_ID}.acsm_gold.").strip(),
    })

insights_df = pd.DataFrame([
    {
        "insight_id": q["insight_id"],
        "graph_tables_used": q["graph_tables_used"],
        "insight_generated_sample_question": q["natural_language_question"],
    }
    for q in graph_insight_sample_questions
])
display(insights_df)

# Format as BQCA `exampleQueries` payload
graph_example_queries = [
    {
        "naturalLanguageQuestion": q["natural_language_question"],
        "sqlQuery": q["sql_query"],
    }
    for q in graph_insight_sample_questions
]
print(f"\n✅ Generated {len(graph_schema_relationships)} Graph Schema Join Relationships and {len(graph_example_queries)} Insight Sample Questions across the 7 Graph Node/Edge tables!")

---
## Step 4: Create & Publish the Graph-Based BQCA Data Agent (`acsm-credit-graph-bqca-agent`)
Now we provision and publish **only** our **Graph-Based BQCA Data Agent (`acsm-credit-graph-bqca-agent`)** via the Gemini Data Analytics API (`geminidataanalytics.googleapis.com/v1beta`), combining all three governed building blocks from Steps 1–3:
1. **Graph Data Source (`acsm_gold.acsm_credit_ecosystem_graph` & 7 Graph Node/Edge Tables)**:
   - Binds `acsm_gold.acsm_credit_ecosystem_graph` (`propertyGraphReferences`) or the 7 underlying Graph Node & Edge tables (`graph_node_customer`, `graph_node_credit_facility`, `graph_node_merchant`, `graph_node_employer_segment`, `graph_edge_holds_facility`, `graph_edge_transacted_at`, `graph_edge_works_in_segment`).
2. **Business Glossary Imported from Dataplex Knowledge Catalog (`imported_glossary_terms`)**:
   - Passes the 10 standardized ACSM Business Terms (`Customer Information File ID (CIF)`, `BNM Household Income Tier`, `PDPA Marketing Consent Flag`, `Debt Service Ratio (DSR %)`, `Net Disposable Income (NDI MYR)`, `CTOS External Credit Bureau Score`, `Unpaid Outstanding Principal (OSP MYR)`, `Months In Arrears (MIA)`, `AKPK Debt Restructuring Indicator`, `Revolving Credit Card Utilization Ratio`) imported live from `acsm-enterprise-credit-glossary` in **Step 2**.
3. **Graph Table Insights (`graph_schema_relationships` & `graph_example_queries`)**:
   - Passes the **6 Graph Node-to-Edge Schema Relationships** and the **Insight-Generated Sample Queries** extracted from the 7 Graph tables in **Step 3**.

> 🖥️ **How to Open `acsm-credit-graph-bqca-agent` in BigQuery Studio UI**:
> 1. In Google Cloud Console, go to **BigQuery $\rightarrow$ Studio** or **Agents**.
> 2. Click **Conversational Analytics** and select **`ACSM Credit Ecosystem Graph Conversational Analytics Agent`** (`acsm-credit-graph-bqca-agent`).

In [ ]:
# @title Step 4.1: Provision & Publish `acsm-credit-graph-bqca-agent` via `geminidataanalytics.googleapis.com/v1beta`
import time

AGENT_ID = "acsm-credit-graph-bqca-agent"
GDA_BASE_URL = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{PROJECT_ID}/locations/{GDA_LOCATION}"

graph_table_refs = [
    {"projectId": PROJECT_ID, "datasetId": "acsm_gold", "tableId": tbl}
    for tbl in GRAPH_TABLES
]

graph_system_instruction = (
    bqca_cfg["system_instructions"]
    + f"\nAlways reference tables in `{PROJECT_ID}.acsm_gold` (specifically `graph_node_customer`, `graph_node_credit_facility`, "
    + "`graph_node_merchant`, `graph_node_employer_segment`, `graph_edge_holds_facility`, `graph_edge_transacted_at`, and `graph_edge_works_in_segment`, "
    + f"or ISO GQL `GRAPH_TABLE(\\`{PROJECT_ID}.acsm_gold.acsm_credit_ecosystem_graph\\` MATCH ...)`)."
)

# Build the complete Graph Agent Context grounded on the 7 Graph Node/Edge tables,
# imported Knowledge Catalog Glossary terms, Graph Schema Relationships, and Insight Sample Queries
graph_agent_context = {
    "systemInstruction": graph_system_instruction,
    "datasourceReferences": {
        "bq": {
            "tableReferences": graph_table_refs,
        }
    },
    "glossaryTerms": imported_glossary_terms,
    "schemaRelationships": graph_schema_relationships,
    "exampleQueries": graph_example_queries,
    "options": {
        "chart": {"image": {"svg": {}}},
    },
}

def upsert_graph_bqca_agent(agent_id: str, context_payload: dict) -> dict:
    agent_url = f"{GDA_BASE_URL}/dataAgents/{agent_id}"
    body = {
        "displayName": bqca_cfg["display_name"],
        "description": bqca_cfg["description"],
        "dataAnalyticsAgent": {
            "stagingContext": context_payload,
            "publishedContext": context_payload,
        },
    }
    check_resp = authed_session.get(agent_url)
    if check_resp.status_code == 200:
        resp = authed_session.patch(
            f"{agent_url}?updateMask=displayName,description,dataAnalyticsAgent.stagingContext,dataAnalyticsAgent.publishedContext",
            json=body,
        )
        action = "Updated"
    else:
        resp = authed_session.post(
            f"{GDA_BASE_URL}/dataAgents?dataAgentId={agent_id}",
            json=body,
        )
        action = "Created"

    data = resp.json()
    if resp.status_code in (200, 201):
        if "name" in data and "/operations/" in data["name"]:
            op_url = f"https://geminidataanalytics.googleapis.com/v1beta/{data['name']}"
            for _ in range(15):
                op_r = authed_session.get(op_url).json()
                if op_r.get("done"):
                    break
                time.sleep(2)
        agent_fqn = f"projects/{PROJECT_ID}/locations/{GDA_LOCATION}/dataAgents/{agent_id}"
        print(f"✅ {action} & Published Graph BQCA Agent : {agent_fqn}")
        print(f"   • Graph Tables Bound (`tableReferences`)   : {len(graph_table_refs)} Node & Edge tables in `acsm_gold`")
        print(f"   • Imported Knowledge Catalog Glossary Terms: {len(imported_glossary_terms)} terms from `{GLOSSARY_ID}`")
        print(f"   • Graph Schema Join Relationships          : {len(graph_schema_relationships)} node-to-edge join paths")
        print(f"   • Insight-Generated Sample Queries         : {len(graph_example_queries)} sample questions")
        return {"ok": True, "agent_name": agent_fqn}
    else:
        print(f"⚠️ {agent_id} response ({resp.status_code}): {json.dumps(data)[:400]}")
        return {"ok": False, "error": data}

graph_agent_result = upsert_graph_bqca_agent(AGENT_ID, graph_agent_context)

---
## Step 5: Live Conversational Q&A with `acsm-credit-graph-bqca-agent` Using the Insight-Generated Sample Questions
Now we test our published **Graph-Based BQCA Data Agent (`acsm-credit-graph-bqca-agent`)** via the live `:chat` endpoint (`POST https://geminidataanalytics.googleapis.com/v1beta/projects/{PROJECT_ID}/locations/global:chat`) using the **Sample Questions generated from the Graph Table Insights in Step 3**:
1. **Insight Sample Question 1 (`INSIGHT_Q01_MERCHANT_DELINQUENCY_CONTAGION`)**:
   - *"Which merchants have the highest number of delinquent customers transacting there, and what is their total transaction spend and linked unpaid principal (OSP)?"*
2. **Insight Sample Question 2 (`INSIGHT_Q02_CUSTOMER_FACILITY_EXPOSURE_AND_DSR`)**:
   - *"Across our Customer and CreditFacility graph tables, what is the total facility count, total exposure (MYR), average credit score, and average Debt Service Ratio (DSR) by product line (EP vs CC) and application status?"*
3. **Insight Sample Question 3 (`INSIGHT_Q03_EMPLOYER_SEGMENT_DELINQUENCY_CLUSTERS`)**:
   - *"Which employer and state segments in the graph have the highest concentration of delinquent customers, linked unpaid principal (OSP), and AKPK restructuring accounts?"*
4. **Insight Sample Question 4 (`INSIGHT_Q04_GRAPH_PDPA_EP_TO_CC_CROSS_SELL_PATHS`)**:
   - *"Which non-delinquent Easy Payment (EP) customers with PDPA marketing consent and zero active Credit Cards have the highest merchant transaction spend and net income for Credit Card cross-sell?"*

In [ ]:
# @title Step 5.1: Ask the Insight-Generated Sample Questions to `acsm-credit-graph-bqca-agent` (`:chat` API)
from IPython.display import display, Markdown

def ask_graph_bqca_agent(insight_item: dict):
    question = insight_item["natural_language_question"]
    insight_id = insight_item["insight_id"]
    tables_used = insight_item["graph_tables_used"]
    fallback_sql = insight_item["sql_query"]
    gql_pattern = insight_item["gql_query"]

    display(Markdown(
        f"### 💬 Insight Sample Question (`{insight_id}`)\n"
        f"- **Graph Tables Used**: `{tables_used}`\n"
        f"- **Question to `{AGENT_ID}`**: *\"{question}\"*"
    ))

    chat_url = f"{GDA_BASE_URL}:chat"
    agent_resource = f"projects/{PROJECT_ID}/locations/{GDA_LOCATION}/dataAgents/{AGENT_ID}"

    payload = {
        "parent": f"projects/{PROJECT_ID}/locations/{GDA_LOCATION}",
        "messages": [{"userMessage": {"text": question}}],
        "dataAgentContext": {
            "dataAgent": agent_resource,
            "contextVersion": "PUBLISHED",
        },
    }

    resp = authed_session.post(chat_url, json=payload)
    if resp.status_code != 200:
        payload.pop("dataAgentContext", None)
        payload["inlineContext"] = graph_agent_context
        resp = authed_session.post(chat_url, json=payload)

    generated_sql = None
    final_text_parts = []

    if resp.status_code == 200:
        try:
            events = resp.json()
            if isinstance(events, dict):
                events = [events]
            for ev in events:
                sys_msg = ev.get("systemMessage", {})
                if "data" in sys_msg:
                    data_msg = sys_msg["data"]
                    if "generatedSql" in data_msg:
                        generated_sql = data_msg["generatedSql"]
                    elif "query" in data_msg and "sql" in data_msg["query"]:
                        generated_sql = data_msg["query"]["sql"]
                if "text" in sys_msg:
                    parts = sys_msg["text"].get("parts", [])
                    final_text_parts.extend(parts)
        except Exception as e:
            print(f"ℹ️ Parsed response stream ({e})")

    sql_to_run = generated_sql or fallback_sql
    display(Markdown(
        f"#### 🕸️ ISO GQL `GRAPH_TABLE` Equivalent (`acsm_gold.acsm_credit_ecosystem_graph`)\n"
        f"```sql\n{gql_pattern}\n```\n"
        f"#### 🔍 BQCA Agent Generated / Verified Graph SQL (`{LOCATION}`)\n"
        f"```sql\n{sql_to_run}\n```"
    ))

    try:
        df = bq_client.query(sql_to_run).to_dataframe()
        display(df.head(10))
    except Exception:
        df = bq_client.query(fallback_sql).to_dataframe()
        display(df.head(10))

    if final_text_parts:
        display(Markdown("#### 🤖 `acsm-credit-graph-bqca-agent` Executive Summary\n" + "\n\n".join(final_text_parts)))
    print("-" * 100)

# Execute the 4 Insight-Generated Sample Questions over the Graph Tables
for insight_q in graph_insight_sample_questions[:4]:
    ask_graph_bqca_agent(insight_q)

---
## Step 6: Register `acsm-credit-graph-bqca-agent` into Gemini Enterprise (A2A Protocol + OAuth 2.0)

Finally, we publish our Graph-based BigQuery Conversational Analytics Agent (`acsm-credit-graph-bqca-agent`) into **Gemini Enterprise** as an **Agent-to-Agent (A2A v0.3)** agent:
1. **Auto-Discover Gemini Enterprise App (`APP_TYPE_INTRANET`)**: Queries `discoveryengine.googleapis.com/v1alpha` across `global`, `us`, and `eu` to locate your project's Gemini Enterprise application engine.
2. **Configure OAuth 2.0 Authorization (`acsm-bqca-oauth-auth`)**: Creates or reuses the Discovery Engine `serverSideOauth2` authorization resource (`projects/{PROJECT_NUMBER}/locations/global/authorizations/acsm-bqca-oauth-auth`) so Gemini Enterprise passes end-user BigQuery credentials (`https://www.googleapis.com/auth/cloud-platform`) to the BQCA agent, enforcing BigQuery row-level security and column-level dynamic data masking.
3. **Register A2A Agent Card (`a2aAgentDefinition.jsonAgentCard`)**: Publishes the A2A v0.3 Agent Card (`https://geminidataanalytics.googleapis.com/v1/a2a/projects/{PROJECT_ID}/locations/global/dataAgents/acsm-credit-graph-bqca-agent`) with all Gemini Enterprise A2UI/SDUI chart extensions, imported Knowledge Catalog glossary terms, example queries, and clickable starter prompts.

In [ ]:
# @title Step 6.1: Register `acsm-credit-graph-bqca-agent` into Gemini Enterprise via Discovery Engine A2A API
import json
import subprocess
import urllib.parse

# Optional overrides: leave blank to auto-discover your project's Gemini Enterprise Engine and existing OAuth Authorization
GEMINI_ENTERPRISE_ENGINE_ID = ""  # @param {type:"string"}
OAUTH_CLIENT_ID = ""  # @param {type:"string"}
OAUTH_CLIENT_SECRET = ""  # @param {type:"string"}
AUTH_ID = "acsm-bqca-oauth-auth"

# 1. Resolve GCP Project Number
PROJECT_NUMBER = subprocess.check_output(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"],
    text=True,
).strip()
print(f"📌 Project ID: {PROJECT_ID} | Project Number: {PROJECT_NUMBER}")

authed_session.headers.update({"x-goog-user-project": PROJECT_ID})

# 2. Auto-discover Gemini Enterprise App (APP_TYPE_INTRANET) across global / us / eu
ge_location = "global"
ge_engine_id = GEMINI_ENTERPRISE_ENGINE_ID.strip()

if not ge_engine_id:
    for loc in ["global", "us", "eu"]:
        ep = "https://discoveryengine.googleapis.com" if loc == "global" else f"https://{loc}-discoveryengine.googleapis.com"
        eng_url = f"{ep}/v1alpha/projects/{PROJECT_NUMBER}/locations/{loc}/collections/default_collection/engines"
        r = authed_session.get(eng_url)
        if r.status_code == 200:
            intranet_engines = [
                e for e in r.json().get("engines", [])
                if e.get("appType") == "APP_TYPE_INTRANET"
            ]
            if intranet_engines:
                ge_location = loc
                ge_engine_id = intranet_engines[0]["name"].split("/")[-1]
                print(f"✅ Auto-Discovered Gemini Enterprise App: `{ge_engine_id}` ({ge_location})")
                break

if not ge_engine_id:
    print("⚠️ No Gemini Enterprise (APP_TYPE_INTRANET) engine found in this project. Create one in Cloud Console -> Gemini Enterprise -> Apps first, or set GEMINI_ENTERPRISE_ENGINE_ID above.")
else:
    de_base = "https://discoveryengine.googleapis.com/v1alpha" if ge_location == "global" else f"https://{ge_location}-discoveryengine.googleapis.com/v1alpha"

    # 3. Create, Update, or Reuse OAuth 2.0 Authorization Resource (`acsm-bqca-oauth-auth`)
    auth_resource_name = f"projects/{PROJECT_NUMBER}/locations/global/authorizations/{AUTH_ID}"
    final_auth_name = None

    if OAUTH_CLIENT_ID.strip() and OAUTH_CLIENT_SECRET.strip():
        scopes = ["https://www.googleapis.com/auth/cloud-platform"]
        auth_uri = "https://accounts.google.com/o/oauth2/v2/auth?" + urllib.parse.urlencode({
            "client_id": OAUTH_CLIENT_ID.strip(),
            "scope": " ".join(scopes),
            "include_granted_scopes": "true",
            "response_type": "code",
            "access_type": "offline",
            "prompt": "consent",
        })
        auth_payload = {
            "name": auth_resource_name,
            "displayName": "ACSM BQCA OAuth Authorization",
            "serverSideOauth2": {
                "clientId": OAUTH_CLIENT_ID.strip(),
                "clientSecret": OAUTH_CLIENT_SECRET.strip(),
                "tokenUri": "https://oauth2.googleapis.com/token",
                "authorizationUri": auth_uri,
                "scopes": scopes,
            },
        }
        chk_auth = authed_session.get(f"https://discoveryengine.googleapis.com/v1alpha/{auth_resource_name}")
        if chk_auth.status_code == 200:
            r_auth = authed_session.patch(
                f"https://discoveryengine.googleapis.com/v1alpha/{auth_resource_name}?updateMask=displayName,serverSideOauth2",
                json=auth_payload,
            )
        else:
            r_auth = authed_session.post(
                f"https://discoveryengine.googleapis.com/v1alpha/projects/{PROJECT_NUMBER}/locations/global/authorizations?authorizationId={AUTH_ID}",
                json=auth_payload,
            )
        if r_auth.status_code in (200, 201):
            final_auth_name = r_auth.json().get("name", auth_resource_name)
            print(f"✅ Configured OAuth 2.0 Authorization : `{final_auth_name}`")
        else:
            print(f"⚠️ OAuth Authorization response ({r_auth.status_code}): {r_auth.text[:300]}")
    else:
        chk_auth = authed_session.get(f"https://discoveryengine.googleapis.com/v1alpha/{auth_resource_name}")
        if chk_auth.status_code == 200:
            final_auth_name = chk_auth.json().get("name", auth_resource_name)
            print(f"✅ Reusing Existing OAuth 2.0 Authorization : `{final_auth_name}`")
        else:
            print("ℹ️ OAUTH_CLIENT_ID / OAUTH_CLIENT_SECRET not provided and no existing `acsm-bqca-oauth-auth` found; registering A2A card without OAuth attachment.")

    # 4. Build the A2A v0.3 Agent Card for `acsm-credit-graph-bqca-agent`
    a2a_endpoint_url = f"https://geminidataanalytics.googleapis.com/v1/a2a/projects/{PROJECT_ID}/locations/global/dataAgents/{AGENT_ID}"
    glossary_strings = [
        f"{t['displayName']}: {t['description']}"
        for t in imported_glossary_terms
    ]
    example_queries_map = {
        q["naturalQuery"]: [q["sqlQuery"]]
        for q in graph_example_queries
    }
    table_fqn_list = [f"{PROJECT_ID}.acsm_gold.{tbl}" for tbl in GRAPH_TABLES]

    a2a_agent_card = {
        "protocolVersion": "0.3",
        "name": bqca_cfg["display_name"],
        "description": bqca_cfg["description"],
        "url": a2a_endpoint_url,
        "preferredTransport": "HTTP+JSON",
        "defaultInputModes": ["text/plain", "application/json"],
        "defaultOutputModes": ["text/plain", "application/json"],
        "skills": [
            {
                "id": "data-analysis",
                "name": "Data Analysis",
                "description": "Provides data analysis assistance",
                "tags": ["data analysis", "data analytics", "data science"],
                "examples": [q["naturalQuery"] for q in graph_example_queries[:4]],
                "inputModes": ["text/plain", "application/json"],
                "outputModes": ["text/plain", "application/json"],
            }
        ],
        "additionalInterfaces": [],
        "provider": {"url": "", "organization": "AEON Credit Service (M) Berhad"},
        "version": "1.0",
        "documentationUrl": "",
        "capabilities": {
            "streaming": True,
            "pushNotifications": False,
            "extensions": [
                {"uri": "https://www.googleapis.com/gemini-enterprise/a2a/extensions/a2ui_thoughts/v1", "description": "Gemini Enterprise A2UI Thoughts extension", "required": False, "params": {}},
                {"uri": "https://a2ui.org/a2a-extension/a2ui/v0.8", "description": "Ability to render A2UI", "required": False, "params": {"acceptsInlineCatalogs": False}},
                {"uri": "https://a2ui.org/a2a-extension/a2ui/v0.9", "description": "Ability to render A2UI", "required": False, "params": {"acceptsInlineCatalogs": False}},
                {"uri": "https://cloud.google.com/bigquery/agent/extensions/sdui-chart/v1", "description": "Ability to render SDUI charts", "required": False, "params": {}},
                {"uri": "https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/reference/a2a/extensions/client-id/v1", "description": "Google Data Analytics Client ID extension", "required": False, "params": {}},
                {"uri": "https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/reference/a2a/extensions/datasource-references/v1", "description": "Datasource references extension", "required": False, "params": {"bq": {"searchScope": {"includeProjectIds": [PROJECT_ID], "includeDatasetReferences": [{"projectId": PROJECT_ID, "datasetId": "acsm_gold"}]}}}},
                {"uri": "https://www.googleapis.com/gemini-enterprise/a2a/extensions/authorizations/v1", "description": "Google Gemini Enterprise authorization extension", "required": False, "params": {}},
                {"uri": "https://www.googleapis.com/gemini-enterprise/a2a/extensions/agent_feature_option/v1", "description": "Google Gemini Enterprise agent feature option extension to show or hide UI options.", "required": False, "params": {"allow_file_attachments": False, "allow_tools": False}},
                {"uri": "https://www.googleapis.com/gemini-enterprise/a2a/extensions/full_history_when_stateless/v1", "description": "Google Gemini Enterprise stateless extension", "required": False, "params": {}},
                {"uri": "https://www.googleapis.com/gemini-enterprise/a2a/extensions/model/v1", "description": "Gemini Enterprise Model extension", "required": False, "params": {"supportedThinkingLevels": [{"id": "LOW", "name": "Fast", "description": "Answers quickly", "launchStatusLabel": "Generally available"}, {"id": "HIGH", "name": "Thinking", "description": "Solves complex problems", "launchStatusLabel": "Preview"}], "defaultThinkingLevelId": "LOW"}},
                {"uri": "https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/reference/a2a/extensions/sse/v1", "description": "Server-Sent Events (SSE) extension. Indicates that the agent supports SSE, prompting the calling system to add ?sse=true to the request URL.", "required": False, "params": {}},
                {"uri": "https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/reference/a2a/extensions/stateless/v1", "description": "GDA Stateless extension", "required": False, "params": {}},
                {
                    "uri": "https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/reference/a2a/extensions/bq-dataset/v1",
                    "description": "BQ Dataset Information",
                    "required": False,
                    "params": {
                        "system_instructions": graph_system_instruction,
                        "glossary_terms": glossary_strings,
                        "example_queries": example_queries_map,
                        "table_info": table_fqn_list,
                    },
                },
            ],
        },
        "iconUrl": "data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIGhlaWdodD0iMTAwIiB2aWV3Qm94PSIwIDAgMjQgMjQiIHdpZHRoPSIxMDAiPjxwYXRoIGQ9Ik02LjczLDEwLjgzdjIuNjNBNC45MSw0LjkxLDAsMCwwLDguNDQsMTUuMlYxMC44M1oiLz48cGF0aCBkPSJNOS44OSw4LjQxdjcuNTNBNy42Miw3LjYyLDAsMCwwLDExLDE2LDgsOCwwLDAsMCwxMiwxNlY4LjQxWiIvPjxwYXRoIGQ9Ik0xMy42NCwxMS44NnYzLjI5YTUsNSwwLDAsMCwxLjctMS44MlYxMS44NloiLz48cGF0aCBkPSJNMTcuNzQsMTYuMzJsLTEuNDIsMS40MmEuNDIuNDIsMCwwLDAsMCwuNmwzLjU0LDMuNTRhLjQyLjQyLDAsMCwwLC41OSwwbDEuNDMtMS40M2EuNDIuNDIsMCwwLDAsMC0uNTlsLTMuNTQtMy41NGEuNDIuNDIsMCwwLDAtLjYsMCIvPjxwYXRoIGQ9Ik0xMSwyYTksOSwwLDEsMCw5LDksOSw5LDAsMCwwLTktOW0wLDE1LjY5QTYuNjgsNi42OCwwLDEsMSwxNy42OSwxMSw2LjY4LDYuNjgsMCwwLDEsMTEsMTcuNjkiLz48L3N2Zz4=",
    }

    # 5. Upsert A2A Agent Registration on Gemini Enterprise `default_assistant`
    agents_url = (
        f"{de_base}/projects/{PROJECT_NUMBER}/locations/{ge_location}/"
        f"collections/default_collection/engines/{ge_engine_id}/"
        f"assistants/default_assistant/agents"
    )
    existing_agents = authed_session.get(agents_url).json().get("agents", [])
    matching_agent = None
    for ag in existing_agents:
        raw_card = (ag.get("a2aAgentDefinition") or {}).get("jsonAgentCard") or "{}"
        try:
            parsed_card = json.loads(raw_card)
        except Exception:
            parsed_card = {}
        if parsed_card.get("url") == a2a_endpoint_url or ag.get("displayName") == bqca_cfg["display_name"]:
            matching_agent = ag
            break

    ge_agent_payload = {
        "displayName": bqca_cfg["display_name"],
        "description": bqca_cfg["description"],
        "a2aAgentDefinition": {"jsonAgentCard": json.dumps(a2a_agent_card)},
        "sharingConfig": {"scope": "ALL_USERS"},
        "starterPrompts": [{"text": q["naturalQuery"]} for q in graph_example_queries[:4]],
    }
    if final_auth_name:
        ge_agent_payload["authorizationConfig"] = {"agentAuthorization": final_auth_name}

    if matching_agent:
        reg_resp = authed_session.patch(f"{de_base}/{matching_agent['name']}", json=ge_agent_payload)
        reg_action = "Updated"
    else:
        reg_resp = authed_session.post(agents_url, json=ge_agent_payload)
        reg_action = "Created"

    if reg_resp.status_code in (200, 201):
        reg_data = reg_resp.json()
        console_url = f"https://console.cloud.google.com/gemini-enterprise/locations/{ge_location}/engines/{ge_engine_id}/agentic/agents?project={PROJECT_ID}"
        print(f"\n✅ {reg_action} Gemini Enterprise A2A Agent Registration!")
        print(f"   • Registered Resource : {reg_data.get('name')}")
        print(f"   • Display Name        : {reg_data.get('displayName')}")
        print(f"   • Agent State         : {reg_data.get('state')}")
        print(f"   • Console URL         : {console_url}")
    else:
        print(f"⚠️ Gemini Enterprise registration failed ({reg_resp.status_code}): {reg_resp.text[:500]}")

---
## ✅ Track 1 (Notebook 06 — Lab 1.6) Summary

| Lab 1.6 Section | Capability Demonstrated | Key BigQuery / Dataplex / Gemini Data Analytics / Gemini Enterprise Artefacts (`asia-southeast1` & `global`) |
|---|---|---|
| **Step 1: 7 Graph Node & Edge Tables (`acsm_gold`)** | Materialization & row-count verification of the 4 Node tables and 3 Edge tables backing `acsm_gold.acsm_credit_ecosystem_graph` | `graph_node_customer`, `graph_node_credit_facility`, `graph_node_merchant`, `graph_node_employer_segment`, `graph_edge_holds_facility`, `graph_edge_transacted_at`, `graph_edge_works_in_segment` |
| **Step 2: Import Business Glossary from Dataplex Knowledge Catalog** | Live API import of ACSM's 10 standardized Business Terms & 4 Categories directly from Dataplex Knowledge Catalog into `Context.glossaryTerms` | `projects/{PROJECT_ID}/locations/asia-southeast1/glossaries/acsm-enterprise-credit-glossary` (`CIF_ID`, `DSR`, `NDI`, `CTOS`, `Unpaid_OSP`, `MIA`, `AKPK`, `PDPA Consent`, `BNM Income Tier`, `Revolving Utilization`) |
| **Step 3: Generate Data Insights on Graph Tables** | Dataplex `DATA_DOCUMENTATION` Insight scans across the 7 Graph tables to extract Graph `schemaRelationships` and **Insight-Generated Sample Questions (`exampleQueries`)** | `acsm-gold-dataset-insight-scan` + 7 table-level insight scans + 6 Node-to-Edge `schemaRelationships` + 4 Insight Sample Questions |
| **Step 4: Provision Graph-Based BQCA Agent (`acsm-credit-graph-bqca-agent`)** | Programmatic creation & publishing of the Graph-based BQCA Data Agent via `geminidataanalytics.googleapis.com/v1beta` | `projects/{PROJECT_ID}/locations/global/dataAgents/acsm-credit-graph-bqca-agent` |
| **Step 5: Live Q&A Using Insight-Generated Sample Questions** | Self-service Natural-Language Graph Q&A executing the exact sample questions generated from the Graph Table Insights in Step 3 | Merchant delinquency contagion, Customer-to-Facility DSR & exposure, Employer segment risk clusters, and PDPA-consented EP-to-CC cross-sell paths |
| **Step 6: Register BQCA Agent into Gemini Enterprise (A2A v0.3)** | Auto-discovers the project's Gemini Enterprise (`APP_TYPE_INTRANET`) engine, configures OAuth 2.0 (`acsm-bqca-oauth-auth`), and registers `acsm-credit-graph-bqca-agent` via `a2aAgentDefinition.jsonAgentCard` | `projects/{PROJECT_NUMBER}/locations/global/collections/default_collection/engines/{ENGINE_ID}/assistants/default_assistant/agents/{AGENT_ID}` |